# 3. Longest Substring Without Repeating Characters

**Difficulty:** Medium  
**Topics:** Hash Table, String, Sliding Window  
**Language:** Python 3  
**Problem:** [LeetCode — Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/)

Find the length of the longest contiguous part of a string whose characters are all different. Start with an exhaustive search, then remove repeated checking with a sliding window and a dictionary of last-seen positions. Each approach includes a dry run, correctness argument, complexity analysis, and complete LeetCode-compatible code.

Run the notebook from top to bottom; only Python's standard library is needed. Edit the JSON string in **Try your own input** to experiment. Copy the final code cell for submission.

## 1. Problem

### Statement summary

For an input string `s`, return the greatest length among its contiguous substrings in which every character occurs at most once. Return an integer length, rather than the substring itself. An empty input has answer `0`.

### Constraints

- `0 <= len(s) <= 100_000`
- Characters may be English letters, digits, symbols, or spaces.

### Examples

**Example 1**

```text
Input: s = "abcabcbb"
Output: 3
```

`"abc"`, `"bca"`, and `"cab"` are contiguous stretches of three distinct characters. No longer stretch satisfies the rule.

**Example 2**

```text
Input: s = "bbbbb"
Output: 1
```

Any two adjacent characters repeat `b`, so the longest valid stretch is a single character.

**Example 3**

```text
Input: s = "pwwkew"
Output: 3
```

`"wke"` is a valid length-three substring. `"pwke"` skips a character in the original string, making it a subsequence rather than an eligible substring.

## 2. Understand the task

A **substring** occupies consecutive positions. We cannot remove interior characters to manufacture a longer answer. Only the length is required, so ties need no special handling.

Character comparisons are case-sensitive: `"aA"` has answer `2`. Spaces, digits, and symbols count as characters; do not strip or filter the input. Python strings are immutable, and neither solution modifies `s`.

Use the supplementary input `"abba"` for both walkthroughs. Its answer is `2`: `"ab"` and `"ba"` qualify. This short input also exposes an important pitfall: when the final `a` arrives, its earlier occurrence is already outside the current window. A correct window must never move its left boundary backward.

An inclusive window from `left` to `right` has length `right - left + 1`. The maximum input length makes checking every substring impractical, motivating a single forward scan.

In [1]:
# Shared supplementary input for both dry runs.
EXAMPLE_S = "abba"
EXAMPLE_EXPECTED = 2

## 3. Approach 1 — exhaustive substring checking

### Idea

Enumerate every nonempty substring. Build a fresh set of its characters; the substring is valid exactly when the set and substring have the same length. Keep the greatest valid length.

This deliberately direct baseline repeats work: overlapping substrings are copied and checked from scratch. It is useful for understanding the contract and checking small inputs, but is unsuitable for the maximum input size.

### Algorithm

1. Initialize `best = 0` to handle the empty string.
2. Choose every start index `left`.
3. Choose every exclusive end index `end` greater than `left`.
4. Copy `candidate = s[left:end]` and check `len(set(candidate)) == len(candidate)`.
5. If valid, update `best`; after all candidates, return it.

### Dry run

`s = "abba"`. Each row groups all candidates from one start index. Every candidate is checked; this implementation does not stop at a duplicate.

| Start | Candidates in checking order | Valid candidates | Best after this row |
|---:|---|---|---:|
| 0 | `a`, `ab`, `abb`, `abba` | `a`, `ab` | 2 |
| 1 | `b`, `bb`, `bba` | `b` | 2 |
| 2 | `b`, `ba` | `b`, `ba` | 2 |
| 3 | `a` | `a` | 2 |

After 10 candidate checks, return `2`.

### Why it works

Every nonempty substring has one start index and one exclusive end index, and the loops enumerate each such pair. A set retains one entry per distinct character, so equal lengths mean no character was repeated. Thus every accepted candidate is valid, and every valid substring is considered. Keeping the maximum accepted length produces the desired answer. If no candidate exists, the initialized `0` is correct.

### Complexity

Let `n = len(s)` and let `k` be the number of distinct characters in `s`.

- **Expected time:** O(n³). There are O(n²) candidates, and copying and building a set costs O(length) per candidate. The sum of all candidate lengths is n(n + 1)(n + 2)/6. Even invalid candidates are fully copied and scanned. Set operations use expected constant time.
- **Auxiliary space:** O(n), from the current substring copy plus at most O(k) set entries. Candidates are processed one at a time, not saved together.

### Implementation

In [2]:
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        best = 0
        for left in range(len(s)):
            for end in range(left + 1, len(s) + 1):
                candidate = s[left:end]
                if len(set(candidate)) == len(candidate):
                    best = max(best, len(candidate))
        return best

BruteForceSolution = Solution
print(BruteForceSolution().lengthOfLongestSubstring(EXAMPLE_S))

2


In [3]:
# Teaching trace: same candidate order and updates as the implementation.
def trace_brute_force(s):
    best = 0
    checks = 0
    for left in range(len(s)):
        for end in range(left + 1, len(s) + 1):
            candidate = s[left:end]
            valid = len(set(candidate)) == len(candidate)
            checks += 1
            if valid:
                best = max(best, len(candidate))
            print(f"Check {checks}: s[{left}:{end}]={candidate!r}, valid={valid}, best={best}")
    print(f"Return {best}; candidate checks = {checks}")
    return best

assert trace_brute_force(EXAMPLE_S) == EXAMPLE_EXPECTED

Check 1: s[0:1]='a', valid=True, best=1
Check 2: s[0:2]='ab', valid=True, best=2
Check 3: s[0:3]='abb', valid=False, best=2
Check 4: s[0:4]='abba', valid=False, best=2
Check 5: s[1:2]='b', valid=True, best=2
Check 6: s[1:3]='bb', valid=False, best=2
Check 7: s[1:4]='bba', valid=False, best=2
Check 8: s[2:3]='b', valid=True, best=2
Check 9: s[2:4]='ba', valid=True, best=2
Check 10: s[3:4]='a', valid=True, best=2
Return 2; candidate checks = 10


## 4. Approach 2 — sliding window with last-seen positions

### From repeated checking to one maintained window

The baseline rebuilds character information for overlapping substrings. Instead, keep a valid window and extend its right boundary once per character. A dictionary `last_seen` maps each character to its latest index in the processed prefix.

When the incoming character previously appeared at index `previous` inside the window, any valid substring ending here must start after `previous`. Jump `left` directly to `previous + 1`. An occurrence before `left` is harmless, so use `max(left, previous + 1)` to prevent moving backward.

The dictionary may retain entries outside the window. Their indices, rather than dictionary membership alone, tell us whether a character conflicts.

### Algorithm

1. Initialize `left = 0`, `best = 0`, and an empty `last_seen` dictionary.
2. Visit each character `char` at index `right`.
3. If `char` was seen, set `left = max(left, last_seen[char] + 1)`.
4. Record `last_seen[char] = right` **after** using the earlier position.
5. Update `best = max(best, right - left + 1)`.
6. Return `best` after the scan.

### Dry run

`s = "abba"`

| `right` | Character | Previous index | `left` before → after | Valid window | `best` |
|---:|---|---|---|---|---:|
| 0 | `a` | absent | 0 → 0 | `a` | 1 |
| 1 | `b` | absent | 0 → 0 | `ab` | 2 |
| 2 | `b` | 1 | 0 → 2 | `b` | 2 |
| 3 | `a` | 0 | 2 → 2 | `ba` | 2 |

The final `a` must not move `left` back to `1`: that would admit `"bba"`, which contains a duplicate. All four characters are processed; return `2`.

### Why it works

After each visit, `s[left:right + 1]` is the **longest valid substring ending at `right`**, and `last_seen` holds each processed character's most recent position.

Initially the window is empty. Suppose the invariant holds before adding a character. Existing window characters are already distinct, so only the incoming character can create a new duplicate. If its previous occurrence is inside the window, moving immediately after that index removes the conflict while keeping every other character distinct. Starting any earlier would retain either that conflict or a duplicate that already prevented an earlier start at the preceding step. If the occurrence is outside the window or absent, the previous boundary remains valid and minimal. The dictionary is then updated to include the current position.

Therefore each step measures the longest valid substring ending at that position. Every nonempty answer ends at some position, so their maximum is the global answer. Empty input leaves `best = 0`.

### Complexity

- **Expected time:** O(n), with one visit per character and expected O(1) dictionary operations. The implementation does not copy window substrings.
- **Auxiliary space:** O(k), where `k <= n` is the number of distinct input characters. Equivalently O(min(n, alphabet size)); the allowed character alphabet bounds dictionary size.

As with sets, pathological hash behavior can worsen runtime; these bounds use the usual expected hash-operation model. Printed teaching traces copy substrings for readability; their cost is separate from the submission algorithm.

### Implementation

In [4]:
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        last_seen = {}
        left = 0
        best = 0
        for right, char in enumerate(s):
            if char in last_seen:
                left = max(left, last_seen[char] + 1)
            last_seen[char] = right
            best = max(best, right - left + 1)
        return best


SlidingWindowSolution = Solution
print(SlidingWindowSolution().lengthOfLongestSubstring(EXAMPLE_S))

2


In [5]:
def trace_sliding_window(s):
    last_seen = {}
    left = 0
    best = 0
    for right, char in enumerate(s):
        previous = last_seen.get(char)
        old_left = left
        if char in last_seen:
            left = max(left, last_seen[char] + 1)
        last_seen[char] = right
        best = max(best, right - left + 1)
        print(f"right={right}, char={char!r}, previous={previous}, "
              f"left={old_left}->{left}, window={s[left:right + 1]!r}, best={best}")
    print(f"Return {best}; character visits = {len(s)}")
    return best

assert trace_sliding_window(EXAMPLE_S) == EXAMPLE_EXPECTED

right=0, char='a', previous=None, left=0->0, window='a', best=1
right=1, char='b', previous=None, left=0->0, window='ab', best=2
right=2, char='b', previous=1, left=0->2, window='b', best=2
right=3, char='a', previous=0, left=2->2, window='ba', best=2
Return 2; character visits = 4


## 5. Comparison and common mistakes

| Approach | Expected time | Auxiliary space | Main trade-off |
|---|---|---|---|
| Exhaustive substring checking | O(n³) | O(n) | Direct, but rebuilds overlapping candidates |
| Sliding window with last-seen positions | O(n) | O(k) | Reuses information; requires careful boundary updates |

Here `n` is input length and `k` is its number of distinct characters. Choose the sliding window for submission: it scales to 100,000 characters and avoids substring copying. Exhaustive checking is a teaching baseline for small inputs.

Common mistakes:

- Returning a substring instead of its length.
- Treating a subsequence as contiguous, as in `"pwke"` for `"pwwkew"`.
- Resetting the whole window on a duplicate, losing useful suffix characters.
- Moving `left` backward for a previous occurrence outside the window (`"abba"`).
- Overwriting a last-seen position before using it to repair the window.
- Using `right - left` instead of `right - left + 1`.
- Removing spaces or merging uppercase and lowercase characters.

The algorithm assumes valid platform input. The manual harness additionally parses JSON and validates constraints; this O(n) validation and its allocations are separate from submission complexity.

## 6. Test the solutions

The first three cases are all official examples. Supplementary cases cover empty input, one character, a backward-boundary trap, overlapping windows, case sensitivity, spaces, and symbols. Test both approaches on these short cases. Also exercise the sliding-window implementation at the maximum length; running exhaustive substring checking at that size would be impractical.

In [6]:
TEST_CASES = [
    ("Problem example 1", "abcabcbb", 3),
    ("Problem example 2", "bbbbb", 1),
    ("Problem example 3", "pwwkew", 3),
    ("Supplementary: empty", "", 0),
    ("Supplementary: one character", "x", 1),
    ("Supplementary: outside-window repeat", "abba", 2),
    ("Supplementary: overlapping answer", "dvdf", 3),
    ("Supplementary: all distinct", "aB3! ", 5),
    ("Supplementary: spaces repeat", "a a", 2),
    ("Supplementary: case matters", "aA", 2),
    ("Supplementary: best is a suffix", "tmmzuxt", 5),
]
for name, s, expected in TEST_CASES:
    for implementation in (BruteForceSolution, SlidingWindowSolution):
        result = implementation().lengthOfLongestSubstring(s)
        assert type(result) is int and result == expected, (name, result, expected)
    print(f"PASS | {name}: {expected}")

# A 100,000-character periodic string: every 95 consecutive characters
# are distinct, and any 96 contain a repeat.
alphabet = "".join(chr(codepoint) for codepoint in range(32, 127))
boundary_s = (alphabet * (100_000 // len(alphabet) + 1))[:100_000]
assert len(boundary_s) == 100_000
assert SlidingWindowSolution().lengthOfLongestSubstring(boundary_s) == 95
print("PASS | Supplementary: maximum length (sliding window): 95")

PASS | Problem example 1: 3
PASS | Problem example 2: 1
PASS | Problem example 3: 3
PASS | Supplementary: empty: 0
PASS | Supplementary: one character: 1
PASS | Supplementary: outside-window repeat: 2
PASS | Supplementary: overlapping answer: 3
PASS | Supplementary: all distinct: 5
PASS | Supplementary: spaces repeat: 2
PASS | Supplementary: case matters: 2
PASS | Supplementary: best is a suffix: 5
PASS | Supplementary: maximum length (sliding window): 95


## 7. Try your own input

Edit `s_text` and optionally `approach`, then rerun the next cell. `s_text` must contain a JSON **string**, including its double quotes: for example, `'"abba"'`. Use `"sliding_window"` for normal inputs or `"brute_force"` for short demonstrations.

The harness accepts 0 to 100,000 printable ASCII characters, covering English letters, digits, symbols, and spaces. It rejects control characters and other alphabets with a clear message; it never trims spaces. As a harness-only safeguard, brute force is limited to 200 characters to keep accidental long runs manageable. The final platform algorithm has no such limit beyond the official constraints. Run All needs no interactive input.

In [7]:
import json

def run_manual(s_text, approach="sliding_window"):
    try:
        s = json.loads(s_text)
        if not isinstance(s, str):
            raise ValueError("Input must be a JSON string, including double quotes.")
        if len(s) > 100_000:
            raise ValueError("s must contain at most 100,000 characters.")
        if any(not 32 <= ord(char) <= 126 for char in s):
            raise ValueError("Use English letters, digits, printable symbols, and spaces.")
        implementations = {
            "brute_force": BruteForceSolution,
            "sliding_window": SlidingWindowSolution,
        }
        if approach not in implementations:
            raise ValueError("approach must be 'brute_force' or 'sliding_window'.")
        if approach == "brute_force" and len(s) > 200:
            raise ValueError("For more than 200 characters, select 'sliding_window'.")
        result = implementations[approach]().lengthOfLongestSubstring(s)
        preview = repr(s) if len(s) <= 60 else repr(s[:57] + "...")
        print(f"{approach}: s={preview}; longest valid substring length = {result}")
        return result
    except (ValueError, TypeError) as error:
        print(f"Input error: {error}")
        return None

# EDIT THESE VALUES, then run this cell again.
s_text = '"abba"'
approach = "sliding_window"  # Or "brute_force" for short inputs.
manual_result = run_manual(s_text, approach)

sliding_window: s='abba'; longest valid substring length = 2


In [8]:
assert run_manual('"dvdf"', "brute_force") == 3
assert run_manual('""') == 0
assert run_manual('" "') == 1
assert run_manual('["a", "b"]') is None
assert run_manual('true') is None
assert run_manual('not JSON') is None
assert run_manual(json.dumps("a\nb")) is None
assert run_manual(json.dumps("a" * 100_001)) is None
assert run_manual('"abc"', "unknown") is None
assert run_manual(json.dumps("a" * 201), "brute_force") is None
print("Manual-input harness checks passed.")

brute_force: s='dvdf'; longest valid substring length = 3
sliding_window: s=''; longest valid substring length = 0
sliding_window: s=' '; longest valid substring length = 1
Input error: Input must be a JSON string, including double quotes.
Input error: Input must be a JSON string, including double quotes.
Input error: Expecting value: line 1 column 1 (char 0)
Input error: Use English letters, digits, printable symbols, and spaces.
Input error: s must contain at most 100,000 characters.
Input error: approach must be 'brute_force' or 'sliding_window'.
Input error: For more than 200 characters, select 'sliding_window'.
Manual-input harness checks passed.


## 8. Final submission

Copy the next cell into LeetCode's Python 3 editor. It implements the required `Solution.lengthOfLongestSubstring(self, s: str) -> int` interface using the sliding window with last-seen positions. No setup helpers or imports are needed.

In [9]:
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        last_seen = {}
        left = 0
        best = 0
        for right, char in enumerate(s):
            if char in last_seen:
                left = max(left, last_seen[char] + 1)
            last_seen[char] = right
            best = max(best, right - left + 1)
        return best